Using distance

In [ ]:
import pymol
from pymol import cmd
import csv
import os

def analyse_pdb_interface(pdb_file, peptide_chain, protein_chain, cutoff=4.0):
    # 1. Load the PDB file into PyMOL
    # 'object_name' will be the filename without the .pdb extension
    object_name = os.path.basename(pdb_file).split('.')[0]
    cmd.load(pdb_file, object_name)
    
    # Define our selection strings based on the chains you provide
    peptide_sel = f"{object_name} and chain {peptide_chain}"
    protein_sel = f"{object_name} and chain {protein_chain}"
    output_csv = f"{object_name}_interactions.csv"

    # 2. Get peptide residues (same logic as before)
    peptide_residues = []
    cmd.iterate(f"{peptide_sel} and name CA", 
                "peptide_residues.append((chain, resi, resn))", 
                space={'peptide_residues': peptide_residues})

    # 3. Open CSV and find neighbors
    with open(output_csv, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Peptide_Residue", "Peptide_Chain", "Protein_Interactions"])

        for chain, resi, resn in peptide_residues:
            current_res_sel = f"({object_name} and chain {chain} and resi {resi})"
            
            # Find protein residues within distance
            interaction_sel = f"({protein_sel}) and (all within {cutoff} of {current_res_sel})"
            
            interacting_res_list = []
            # We use 'byres' to get the full identity of the neighbor
            cmd.iterate(f"byres {interaction_sel} and name CA", 
                        "interacting_res_list.append(f'{resn}{resi}_{chain}')", 
                        space={'interacting_res_list': interacting_res_list})
            
            interactions_str = "; ".join(sorted(set(interacting_res_list)))
            writer.writerow([f"{resn}{resi}", chain, interactions_str])

    print(f"Success! Created {output_csv}")

# Example of how you would run it:
# analyse_pdb_interface("my_complex.pdb", "B", "A")

In [3]:
analyse_pdb_interface('7TA4.pdb', "D", "B")

Success! Created 7TA4_interactions.csv


Using plip

In [ ]:
from plip.structure.preparation import PDBComplex
import pandas as pd

def run_plip_analysis(pdb_path, peptide_chain):
    my_mol = PDBComplex()
    my_mol.load_pdb(pdb_path)
    
    # PLIP automatically finds ligands, but for a peptide, 
    # we define it by its chain ID
    my_mol.characterize_complex()
    
    results = []
    
    # PLIP stores interactions in 'interaction_sets'
    # We filter for interactions involving your peptide chain
    for key in my_mol.interaction_sets:
        # Check if the interaction involves your peptide
        if peptide_chain in key:
            inter_set = my_mol.interaction_sets[key]
            
            # Gather all types of interactions
            all_interactions = inter_set.hbonds + inter_set.saltbridge + \
                               inter_set.hydrophobic_contacts + inter_set.pistacking
            
            for inter in all_interactions:
                results.append({
                    "Peptide_Res": f"{inter.restype_l}{inter.resnr_l}",
                    "Protein_Res": f"{inter.restype_p}{inter.resnr_p}",
                    "Type": inter.__class__.__name__,
                    "Distance": getattr(inter, 'dist', 'N/A')
                })

    df = pd.DataFrame(results)
    df.to_csv("plip_results.csv", index=False)
    return df

# run_plip_analysis("my_file.pdb", "B")

In [18]:
run_plip_analysis("7TA4.pdb", "D")

""


In [43]:
from plip.structure.preparation import PDBComplex, LigandFinder

mol = PDBComplex()
mol.load_pdb("7TA4.pdb")
lfinder = LigandFinder(mol.protcomplex, mol.altconf, mol.modres, mol.covalent, mol.Mapper)
peptide_ligand_obj = lfinder.getpeptides("D")

if peptide_ligand_obj:
    mol.ligands.append(peptide_ligand_obj)
    mol.characterize_complex(peptide_ligand_obj)
    mol.interaction_sets[peptide_ligand_obj]
else:
    print("could not find residues in chain D")


TypeError: unhashable type: 'list'

In [ ]:
from plip.structure.preparation import PDBComplex, LigandFinder, PLInteraction
from plip.exchange.report import BindingSiteReport

def force_peptide_as_ligand(pdb_path, peptide_chain_id):
    mol = PDBComplex()
    mol.load_pdb(pdb_path)
    
    # 1. Manually find the peptide residues and bundle them as a Ligand object
    lfinder = LigandFinder(mol.protcomplex, mol.altconf, mol.modres, mol.covalent, mol.Mapper)
    peptide_ligand_obj = lfinder.getpeptides(peptide_chain_id)
    
    if peptide_ligand_obj:
        
        # 3. This is the "Engine" class you were looking for. 
        # It takes the ligand object and the protein complex as input.
        interactions = PLInteraction(peptide_ligand_obj, mol.protcomplex)
        
        # 4. Use BindingSiteReport to parse the raw geometric data into H-bonds, etc.
        report = BindingSiteReport(interactions)
        
        # 5. Manually populate the dictionary so you can access it like the web version
        mol.interaction_sets[peptide_ligand_obj.longname] = report
        
        print(f"Successfully characterized peptide chain {peptide_chain_id}")
        return report
    else:
        print(f"Could not extract peptide from chain {peptide_chain_id}")
        return None

In [56]:
force_peptide_as_ligand("7TA4.pdb", "D")

TypeError: PLInteraction.__init__() missing 1 required positional argument: 'protcomplex'

In [51]:
from plip.structure.preparation import PDBComplex

def analyze_peptide_with_plip(pdb_path, peptide_chain_id):
    mol = PDBComplex()
    mol.load_pdb(pdb_path)
    
    # This is the 'magic' specification call.
    # The ":" prefix tells PLIP to treat the entire chain as a ligand.
    peptide_key = f":{peptide_chain_id}"
    
    # This replaces the generic mol.analyze()
    mol.characterize_complex(peptide_key)
    
    # The results are stored in interaction_sets using the same key
    if peptide_key in mol.interaction_sets:
        interactions = mol.interaction_sets[peptide_key]
        
        # Now you can access specific types like the web version:
        print(f"Hydrophobic contacts: {len(interactions.hydrophobic_contacts)}")
        print(f"Hydrogen bonds: {len(interactions.hbonds)}")
        
        # Example: Loop through H-bonds
        for hb in interactions.hbonds:
            print(f"Peptide {hb.restype_l}{hb.resnr_l} -> Protein {hb.restype_p}{hb.resnr_p}")
    else:
        print(f"No interactions found for chain {peptide_chain_id}")

# Usage:
# analyze_peptide_with_plip("my_complex.pdb", "B")

In [52]:
analyze_peptide_with_plip("7TA4.pdb", "D")

AttributeError: 'str' object has no attribute 'members'